In [2]:
import pandas as pd
import re
from collections import defaultdict, Counter

df = pd.read_csv("MOVIE_REVIEWS.csv")
df.head()


,Review,Rating
0,Totally waste of time and money worst movie to...,1
1,such an awful movie I have watched in a while,1
2,Worst shark film ever and did not think sharks...,1
3,The plot per se was stupid,2
4,I am very impressed how a mouth to mouth under...,2


In [3]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["Cleaned"] = df["Review"].apply(clean_text)
df["Tokens"] = df["Cleaned"].apply(lambda x: x.split())

df.head()


,Review,Rating,Cleaned,Tokens
0,Totally waste of time and money worst movie to...,1,totally waste of time and money worst movie to...,"[totally, waste, of, time, and, money, worst, ..."
1,such an awful movie I have watched in a while,1,such an awful movie i have watched in a while,"[such, an, awful, movie, i, have, watched, in,..."
2,Worst shark film ever and did not think sharks...,1,worst shark film ever and did not think sharks...,"[worst, shark, film, ever, and, did, not, thin..."
3,The plot per se was stupid,2,the plot per se was stupid,"[the, plot, per, se, was, stupid]"
4,I am very impressed how a mouth to mouth under...,2,i am very impressed how a mouth to mouth under...,"[i, am, very, impressed, how, a, mouth, to, mo..."


In [5]:
class_word_counts = defaultdict(Counter)
class_doc_counts = Counter()

for _, row in df.iterrows():
    rating = row["Rating"]
    class_doc_counts[rating] += 1
    class_word_counts[rating].update(row["Tokens"])


In [6]:
total_docs = len(df)
priors = {c: class_doc_counts[c] / total_docs for c in class_doc_counts}

priors


{1: 0.14, 2: 0.2, 3: 0.18, 4: 0.24, 5: 0.24}

In [8]:
# Build global vocabulary from all classes
global_vocab = set()
for c in class_word_counts:
    global_vocab.update(class_word_counts[c].keys())

# Test document
test_doc = "watched this beautiful and adventure filled film just now and it was the best film ever"
test_clean = clean_text(test_doc)
test_tokens = test_clean.split()

# Drop unknown words
updated_test_tokens = [w for w in test_tokens if w in global_vocab]
updated_test_tokens


['watched',
 'this',
 'beautiful',
 'and',
 'adventure',
 'filled',
 'film',
 'just',
 'and',
 'it',
 'was',
 'the',
 'film',
 'ever']

In [9]:
def likelihood(word, rating):
    vocab_size = len(global_vocab)
    word_count = class_word_counts[rating][word]
    total_words = sum(class_word_counts[rating].values())
    return (word_count + 1) / (total_words + vocab_size)


In [10]:
scores = {}

for c in class_word_counts:
    score = priors[c]
    for w in updated_test_tokens:
        score *= likelihood(w, c)
    scores[c] = score

scores


{1: 3.8857805861880497e-32,
 2: 3.7263853219749853e-31,
 3: 5.3862533431046995e-33,
 4: 8.774300396252768e-32,
 5: 1.2747693903788626e-30}

In [11]:
predicted_class = max(scores, key=scores.get)
predicted_class


5

In [12]:
print("1. Prior probabilities:")
print(priors, "\n")

print("2. Updated words after dropping unknown words:")
print(updated_test_tokens, "\n")

print("3. Likelihood probabilities for each class:")
for c in class_word_counts:
    print(f"Class {c}:")
    for w in updated_test_tokens:
        print(f"  P({w}|{c}) = {likelihood(w, c)}")
    print()

print("4. Final scores for all classes:")
print(scores, "\n")

print("5. Predicted class label:")
print(predicted_class)


1. Prior probabilities:
{1: 0.14, 2: 0.2, 3: 0.18, 4: 0.24, 5: 0.24} 

2. Updated words after dropping unknown words:
['watched', 'this', 'beautiful', 'and', 'adventure', 'filled', 'film', 'just', 'and', 'it', 'was', 'the', 'film', 'ever'] 

3. Likelihood probabilities for each class:
Class 1:
  P(watched|1) = 0.00625
  P(this|1) = 0.0125
  P(beautiful|1) = 0.003125
  P(and|1) = 0.0125
  P(adventure|1) = 0.003125
  P(filled|1) = 0.003125
  P(film|1) = 0.0125
  P(just|1) = 0.003125
  P(and|1) = 0.0125
  P(it|1) = 0.003125
  P(was|1) = 0.003125
  P(the|1) = 0.0125
  P(film|1) = 0.0125
  P(ever|1) = 0.0125

Class 2:
  P(watched|2) = 0.002583979328165375
  P(this|2) = 0.012919896640826873
  P(beautiful|2) = 0.002583979328165375
  P(and|2) = 0.023255813953488372
  P(adventure|2) = 0.002583979328165375
  P(filled|2) = 0.002583979328165375
  P(film|2) = 0.007751937984496124
  P(just|2) = 0.007751937984496124
  P(and|2) = 0.023255813953488372
  P(it|2) = 0.0103359173126615
  P(was|2) = 0.00775